Requirements:

1. The result script should read all the file in `./image` and `./mask` and apply your algorithm to classify.

2. The script needs to save `output.csv` in this folder, which should contain two columns: `image_id` (save the name of test sample in `./image` ) and `dx` (save the classification result for thee corresponding image, the options are: mel, nv, and vasc). (The format is the same as provided `label.csv`)

In [18]:
# ======================================================
# 成员 1 —— 数据与预处理模块
# 功能：数据集整理 + 路径管理 + 图像预处理 + 划分训练/测试集
# ======================================================

import os
import cv2
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# ====================== 1. 路径配置（全自动适配你的项目） ======================
BASE_DIR = "../Data_Proj2"
IMAGE_DIR = os.path.join(BASE_DIR, "image")
MASK_DIR = os.path.join(BASE_DIR, "mask")
LABEL_PATH = os.path.join(BASE_DIR, "label.csv")

OUTPUT_DIR = "./processed_dataset"  # 预处理后输出文件夹
TRAIN_DIR = os.path.join(OUTPUT_DIR, "train")
TEST_DIR = os.path.join(OUTPUT_DIR, "test")

# 创建输出文件夹
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

# 读取标签
df = pd.read_csv(LABEL_PATH)
print("✅ 标签读取完成，共 {} 条数据".format(len(df)))

# ====================== 2. 图像预处理函数 ======================
def preprocess_image(img_path, target_size=256):
    img = cv2.imread(img_path)
    if img is None:
        return None

    # 1. 尺寸归一化
    img = cv2.resize(img, (target_size, target_size))

    # 2. 轻度去毛发
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 9))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)

    _, hair_mask = cv2.threshold(blackhat, 12, 255, cv2.THRESH_BINARY)

    img = cv2.inpaint(img, hair_mask, 1, cv2.INPAINT_TELEA)

    # 3. 轻度亮度归一化，不要过度增强
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    l = cv2.normalize(l, None, 30, 220, cv2.NORM_MINMAX)

    lab = cv2.merge((l, a, b))
    img = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

    return img

# ====================== 3. 划分训练集 / 测试集 ======================
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["dx"]
)

print("✅ 训练集数量：", len(train_df))
print("✅ 测试集数量：", len(test_df))

# ====================== 4. 批量预处理并保存 ======================
def process_and_save(df, save_dir):
    for idx, row in df.iterrows():
        image_id = row["image_id"]
        img_path = os.path.join(IMAGE_DIR, f"{image_id}.jpg")
        img = preprocess_image(img_path)

        if img is not None:
            save_path = os.path.join(save_dir, f"{image_id}.jpg")
            cv2.imwrite(save_path, img)

process_and_save(train_df, TRAIN_DIR)
process_and_save(test_df, TEST_DIR)

# ====================== 5. 保存划分后的标签 ======================
train_df.to_csv(os.path.join(OUTPUT_DIR, "train_label.csv"), index=False)
test_df.to_csv(os.path.join(OUTPUT_DIR, "test_label.csv"), index=False)

print("="*60)
print("🎉 数据预处理全部完成！")
print("📁 输出路径：", OUTPUT_DIR)
print("├─ 训练集：", TRAIN_DIR)
print("├─ 测试集：", TEST_DIR)
print("├─ train_label.csv")
print("└─ test_label.csv")
print("✅ 已输出可直接用于训练的标准数据集！")

✅ 标签读取完成，共 600 条数据
✅ 训练集数量： 480
✅ 测试集数量： 120
🎉 数据预处理全部完成！
📁 输出路径： ./processed_dataset
├─ 训练集： ./processed_dataset\train
├─ 测试集： ./processed_dataset\test
├─ train_label.csv
└─ test_label.csv
✅ 已输出可直接用于训练的标准数据集！
